# VitalCheck: a ward vitals monitor
**Sample mini-project · Track A (Patient Vitals Monitor) · MED23CL202 Python Programming Laboratory**

A nurse records each patient's temperature, pulse and blood pressure. VitalCheck flags abnormal readings at once, uses stricter limits for ICU patients, keeps every reading in a file in Google Drive, and shows a summary and charts.

This is a **worked example** of a complete mini-project. It uses one idea from every experiment (1 to 10), so you can see how they fit together in one program. Your own project can be on any topic.

**Before you start:** rename this notebook with your register number (click the name at the top left), e.g. `E0125001_miniproject.ipynb`. Keep that title bar visible in your screenshots.

**How to run it:** run the cells from top to bottom with **Shift + Enter**. Cell 9 fills in some sample readings so you can see everything working; cell 10 is the full menu program you type into.

> The normal limits used here are simplified teaching values for practice. This is a student project, not a medical device.

## 1. Setup: imports and where the data file lives *(Experiments 2, 5 and 6)*
The readings are saved in **Google Drive**, so they are still there tomorrow. If Drive isn't connected, the file goes into Colab's own folder, which is wiped when the runtime disconnects.

In [ ]:
# Cell 1: modules, and the data file's location
import os
import csv
import statistics
from datetime import datetime

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = "/content/drive/MyDrive/MiniProject"      # permanent: Google Drive
except Exception:
    DATA_DIR = "vitals_data"                              # not in Colab, or Drive not connected
    print("Google Drive is not connected; saving in a local folder instead.")

os.makedirs(DATA_DIR, exist_ok=True)
DATA_FILE = os.path.join(DATA_DIR, "vitals.csv")
print("Readings are saved in:", DATA_FILE)

## 2. Limits and my own exception *(Experiments 1 and 8)*
A dictionary holds the limits for each vital sign. `InvalidReadingError` is **my own exception**: it is raised for a value no living patient can have (usually a typing mistake).

In [ ]:
# Cell 2: limits (dictionaries) and a user-defined exception

# Values outside these are impossible, so they are rejected as typing mistakes.
POSSIBLE = {"temp": (85.0, 115.0), "pulse": (20, 250), "systolic": (50, 260)}

# Simplified normal limits for an adult (teaching values, not clinical advice).
NORMAL = {"temp": (97.0, 100.3), "pulse": (60, 100), "systolic": (90, 139)}

# Stricter limits for ICU patients: smaller changes need attention sooner.
ICU_NORMAL = {"temp": (97.0, 99.4), "pulse": (60, 90), "systolic": (100, 129)}

UNITS = {"temp": "°F", "pulse": "beats/min", "systolic": "mmHg"}


class InvalidReadingError(Exception):
    """Raised when a vital sign is outside what any living patient can have."""
    pass


def check_possible(name, value):
    """Raise InvalidReadingError if a value is impossible; otherwise return it."""
    low, high = POSSIBLE[name]
    if value < low or value > high:
        raise InvalidReadingError(f"{name} {value} {UNITS[name]} is not possible (allowed {low}–{high}).")
    return value

## 3. Classes: Patient and ICUPatient *(Experiments 3 and 4)*
`Patient` keeps its readings in a **private** list (`__readings`) that can only change through `add_reading()`, which checks every value first.
`ICUPatient` **inherits** from `Patient` and **overrides** `limits()`, so the same `status()` call gives stricter results for ICU patients (**polymorphism**).

In [ ]:
# Cell 3: classes, encapsulation, inheritance and overriding

class Patient:
    """A patient on the ward and their vital-sign readings."""

    def __init__(self, name, bed):
        self.name = name
        self.bed = bed
        self.__readings = []          # private: changed only through add_reading()

    def add_reading(self, temp, pulse, systolic, time=None):
        """Check a reading, then store it. Raises InvalidReadingError for impossible values."""
        check_possible("temp", temp)
        check_possible("pulse", pulse)
        check_possible("systolic", systolic)
        reading = {"time": time or datetime.now().strftime("%Y-%m-%d %H:%M"),
                   "temp": temp, "pulse": pulse, "systolic": systolic}
        self.__readings.append(reading)
        return reading

    def get_readings(self):
        """Return a copy, so outside code can't change the stored readings."""
        return list(self.__readings)

    def kind(self):
        return "Ward"

    def limits(self):
        """The normal limits used for this patient."""
        return NORMAL

    def status(self, reading):
        """Return a list of alerts for one reading (an empty list means all normal)."""
        alerts = []
        for name, (low, high) in self.limits().items():
            value = reading[name]
            if value > high:
                alerts.append(f"{name} HIGH ({value} {UNITS[name]})")
            elif value < low:
                alerts.append(f"{name} LOW ({value} {UNITS[name]})")
        return alerts


class ICUPatient(Patient):
    """An ICU patient: the same checks, with stricter limits."""

    def kind(self):
        return "ICU"

    def limits(self):                 # overrides Patient.limits()
        return ICU_NORMAL

## 4. Helper functions: safe input and the alert label *(Experiments 1, 2 and 7)*
`read_number()` keeps asking until the nurse types a real number, so wrong input never crashes the program.

In [ ]:
# Cell 4: input that can't crash the program, and a function that returns a label

def read_number(prompt, kind=float):
    """Ask until the user types a number; return it as an int or a float."""
    while True:
        try:
            if kind == int:
                return int(input(prompt))
            return float(input(prompt))
        except ValueError:
            print("  Please type a number, e.g. 98.6")


def alert_label(alerts):
    """Turn a list of alerts into one short label."""
    if not alerts:
        return "NORMAL"
    elif len(alerts) == 1:
        return "CHECK: " + alerts[0]
    else:
        return "URGENT: " + "; ".join(alerts)


def fahrenheit(celsius):
    """Convert a Celsius thermometer reading to °F."""
    return round(celsius * 9 / 5 + 32, 1)

## 5. Saving and loading the readings file *(Experiments 5 and 6)*
Every reading is **appended** to a CSV file (mode `"a"`), so nothing already saved is lost. When the program starts, it reads the file back and rebuilds the patients.

In [ ]:
# Cell 5: file handling (CSV, append and read)

FIELDS = ["time", "name", "bed", "kind", "temp", "pulse", "systolic"]


def save_reading(patient, reading, quiet=False):
    """Append one reading to the CSV file, writing the header the first time."""
    new_file = not os.path.exists(DATA_FILE)
    try:
        with open(DATA_FILE, "a", newline="") as f:
            writer = csv.writer(f)
            if new_file:
                writer.writerow(FIELDS)
            writer.writerow([reading["time"], patient.name, patient.bed, patient.kind(),
                             reading["temp"], reading["pulse"], reading["systolic"]])
    except OSError as e:
        print("Could not save the reading:", e)
    finally:
        if not quiet:
            print(f"  (file: {DATA_FILE})")


def load_patients():
    """Read the CSV file and rebuild every patient with their readings."""
    patients = {}
    if not os.path.exists(DATA_FILE):
        return patients
    with open(DATA_FILE, "r", newline="") as f:
        reader = csv.reader(f)
        next(reader)                                  # skip the header row
        for row in reader:
            time, name, bed, kind, temp, pulse, systolic = row
            if name not in patients:
                cls = ICUPatient if kind == "ICU" else Patient
                patients[name] = cls(name, bed)
            try:
                patients[name].add_reading(float(temp), int(pulse), int(systolic), time)
            except (ValueError, InvalidReadingError):
                print("  Skipped a damaged row:", row)
    return patients

## 6. Analysis with pandas *(Experiment 9)*

In [ ]:
# Cell 6: a pandas summary of all readings

def summary():
    """Print each patient's average, lowest and highest readings, and every fever."""
    if not os.path.exists(DATA_FILE):
        print("No readings saved yet.")
        return None
    df = pd.read_csv(DATA_FILE)
    print("Readings saved:", df.shape[0])
    print()
    table = df.groupby("name")[["temp", "pulse", "systolic"]].agg(["mean", "min", "max"]).round(1)
    print(table)
    fevers = df[df["temp"] >= 100.4]
    print()
    print(f"Fever readings (100.4 °F or more): {len(fevers)}")
    if len(fevers) > 0:
        print(fevers[["time", "name", "temp"]].to_string(index=False))
    return df

## 7. Charts *(Experiment 10)*

In [ ]:
# Cell 7: a labelled line chart for one patient, and a histogram of all temperatures

def chart_patient(df, name):
    """Line chart of one patient's temperature over time."""
    rows = df[df["name"] == name]
    if rows.empty:
        print("No readings for", name)
        return
    plt.figure(figsize=(7, 3.5))
    plt.plot(range(1, len(rows) + 1), rows["temp"], marker="o")
    plt.axhline(100.4, color="red", linestyle="--", label="Fever (100.4 °F)")
    plt.xticks(range(1, len(rows) + 1))
    plt.xlabel("Reading number")
    plt.ylabel("Temperature (°F)")
    plt.title(f"Temperature of {name}")
    plt.legend()
    plt.tight_layout()
    plt.show()


def chart_all(df):
    """Histogram of every temperature on the ward."""
    plt.figure(figsize=(7, 3.5))
    sns.histplot(data=df, x="temp", bins=8)
    plt.gca().yaxis.get_major_locator().set_params(integer=True)
    plt.xlabel("Temperature (°F)")
    plt.ylabel("Number of readings")
    plt.title("All temperatures on the ward")
    plt.tight_layout()
    plt.show()

## 8. One report for every patient *(polymorphism, Experiment 4)*
The same line, `p.status(last)`, works for ward and ICU patients alike. Inside `status()`, the call `self.limits()` runs `Patient`'s version or `ICUPatient`'s version depending on which object it is, so an ICU patient is flagged sooner. `p.kind()` in the printed line works the same way. One call, different behaviour: that is **polymorphism**.

In [ ]:
# Cell 8: the ward round, one line for every kind of patient

def ward_round(patients):
    """Print the latest reading and alert label for every patient."""
    if not patients:
        print("No patients yet.")
        return
    temps = []
    for p in patients.values():
        readings = p.get_readings()
        if not readings:
            print(f"{p.name:10} bed {p.bed:6} {p.kind():4}  no readings")
            continue
        last = readings[-1]
        temps.append(last["temp"])
        print(f"{p.name:10} bed {p.bed:6} {p.kind():4}  {last['temp']} °F  {last['pulse']} bpm  "
              f"{last['systolic']} mmHg  ->  {alert_label(p.status(last))}")
    if temps:
        print(f"\nAverage latest temperature: {statistics.mean(temps):.1f} °F")

## 9. Try it with sample readings
This cell adds a few readings without typing, so you can see the whole program work. Run it once; running it again adds the same readings again.

In [ ]:
# Cell 9: sample readings (no typing needed)

patients = load_patients()
samples = [
    (Patient, "Ravi", "12", [(98.6, 78, 118), (99.1, 82, 121), (100.8, 96, 128)]),
    (ICUPatient, "Meena", "ICU-3", [(98.9, 84, 124), (99.6, 92, 131)]),
    (Patient, "Sara", "14", [(97.9, 70, 112)]),
]
for cls, name, bed, readings in samples:
    if name not in patients:
        patients[name] = cls(name, bed)
    for temp, pulse, systolic in readings:
        reading = patients[name].add_reading(temp, pulse, systolic)
        save_reading(patients[name], reading, quiet=True)
print("Sample readings saved in", DATA_FILE)

# An impossible value is refused by my own exception:
try:
    patients["Sara"].add_reading(150, 70, 112)
except InvalidReadingError as e:
    print("Refused:", e)

print()
ward_round(patients)

## 10. The VitalCheck program
Run this cell and follow the menu. Type **0** to finish.

In [ ]:
# Cell 10: the menu program

# `main` is just a name we chose: it is an ordinary function (Module 8) that holds the menu program.
# Defining it does not run it; the last line, main(), calls it and starts the menu.
def main():
    patients = load_patients()
    print(f"VitalCheck: {len(patients)} patient(s) loaded from the file.")
    while True:
        print("\n1 Add a patient   2 Add a reading   3 Ward round   4 Summary   5 Charts   0 Quit")
        choice = input("Choose: ").strip()
        if choice == "1":
            name = input("Patient name: ").strip().title()
            bed = input("Bed: ").strip()
            icu = input("ICU patient? (y/n): ").strip().lower() == "y"
            patients[name] = ICUPatient(name, bed) if icu else Patient(name, bed)
            print(f"Added {name} ({patients[name].kind()}).")
        elif choice == "2":
            name = input("Patient name: ").strip().title()
            if name not in patients:
                print("No such patient. Add them first (option 1).")
                continue
            unit = input("Temperature in C or F? ").strip().upper()
            temp = read_number("Temperature: ")
            if unit == "C":
                temp = fahrenheit(temp)
            pulse = read_number("Pulse (beats/min): ", int)
            systolic = read_number("Systolic BP (mmHg): ", int)
            try:
                reading = patients[name].add_reading(temp, pulse, systolic)
            except InvalidReadingError as e:
                print("Not saved:", e)
            else:
                save_reading(patients[name], reading)
                print(name, "->", alert_label(patients[name].status(reading)))
        elif choice == "3":
            ward_round(patients)
        elif choice == "4":
            summary()
        elif choice == "5":
            df = summary()
            if df is not None:
                chart_patient(df, input("Chart for which patient? ").strip().title())
                chart_all(df)
        elif choice == "0":
            print("Goodbye. All readings are saved in", DATA_FILE)
            break
        else:
            print("Please choose 0 to 5.")


main()

## How this project uses the course
| Experiment | Where in this notebook |
|---|---|
| 1. Data types, operators, decisions | `float(input(...))` and `int(input(...))` in `read_number()`; `check_possible()`, `status()`, `fahrenheit()` |
| 2. Loops, collections, functions | `while True` in `read_number()` and `main()`; `for` in `ward_round()`; lists and dictionaries; `statistics` module |
| 3. Classes and encapsulation | `Patient` with private `__readings`, `add_reading()` and `get_readings()` |
| 4. Inheritance and polymorphism | `ICUPatient(Patient)` overrides `limits()` and `kind()`; `self.limits()` in `status()` and `p.kind()` in `ward_round()` |
| 5 and 6. Files | `save_reading()` appends to a CSV; `load_patients()` reads it back |
| 7. try / except | `read_number()`; `except (ValueError, InvalidReadingError)` in `load_patients()` |
| 8. raise, finally, own exception | `InvalidReadingError`; `raise` in `check_possible()`; `finally` in `save_reading()` |
| 9. pandas | `summary()`: `read_csv`, `groupby`, a fever filter |
| 10. Charts | `chart_patient()` (Matplotlib) and `chart_all()` (Seaborn) |

**References**
1. Python Software Foundation, "The Python Tutorial," docs.python.org/3/tutorial.
2. pandas development team, "pandas documentation," pandas.pydata.org.
3. Matplotlib and seaborn documentation, matplotlib.org and seaborn.pydata.org.
4. MED23CL202 Python Lab Manual and the Foothold Python course (Experiments 1–10).